# Gemini Embedding 2 & Vector Search 2.0 Workshop 🔑

---

# [Part 1] 멀티모달 임베딩부터 Vector Search 2.0 검색까지 (약 30분)

이 노트북은 하나의 원칙으로 구성되어 있습니다.

> **로컬 연산은 원리를 보여주는 glass box, 최종 결과는 Vector Search 2.0이 반환한다.**

즉, NumPy로 직접 짠 코사인 유사도와 BM25는 "검색 엔진 내부에서 실제로 일어나는 연산"을 열어 보이는 교육 장치이고,
**같은 질의를 매니지드 서비스(Vector Search 2.0)로 다시 실행해 대조**합니다.

* **1단계** 비디오 전처리 및 고속 청킹 (FFmpeg Stream Copy)
* **2단계** Dense 임베딩 · 코사인 유사도 · t-SNE 시간축 궤적 · Dense 단독 크로스모달 검색
* **3단계** 로컬 하이브리드 검색 (SimpleBM25 + `alpha` 가중치)
* **4단계** Vector Search 2.0 — kNN 검색 / 내장 RRF 하이브리드 / 메타데이터 필터
* **5단계** 검색 결과 최적화 (Ranking API 리랭킹 + 비디오 크라우딩 필터)
* **6단계** 자원 정리

> 손으로 짠 코사인 유사도 = 벡터 N개에 대한 O(n) 완전탐색.
> Vector Search 2.0 = 동일한 수학을, 매니지드 서버리스 엔진에서, 인프라 없이.

> [!WARNING]
> **패키지 설치 및 자동 커널 재시작 안내**
> * 아래 설치 셀을 실행하면 실습에 필요한 라이브러리가 자동으로 설치됩니다.
> * 설치가 끝나면, 업데이트된 바이너리 패키지(NumPy, SciPy 등)를 메모리에 안전하게 로드하기 위해 🚨주피터 커널이 1회 자동으로 재시작🚨됩니다.
> * 커널 상태가 초기화되는 것은 정상적인 흐름입니다. 당황하지 마시고, 재시작이 끝나면(커널 상태가 `idle`) **설치 셀부터 다시** 순서대로 실행해 주세요.
> * 두 번째 실행부터는 `✅ 커널 재시작이 이미 완료된 세션입니다` 메시지만 나오고 넘어갑니다.

In [ ]:
# Install necessary libraries (최초 1회, 약 40초)
%pip install -qU google-genai google-cloud-vectorsearch google-cloud-discoveryengine google-cloud-storage Pillow opencv-python imageio-ffmpeg numpy scikit-learn matplotlib seaborn

# NumPy/SciPy 같은 바이너리 패키지는 업그레이드해도 "이미 메모리에 올라간 구버전"이 계속 쓰입니다.
# import가 성공하더라도 뒤쪽 셀에서 ABI 불일치로 깨지므로, 설치 직후 커널을 무조건 1회 재시작합니다.
import os
import IPython

_RESTART_FLAG = "/tmp/.smx_workshop_kernel_restarted"

if os.path.exists(_RESTART_FLAG):
    print("✅ 커널 재시작이 이미 완료된 세션입니다. 다음 셀로 진행하세요.")
else:
    with open(_RESTART_FLAG, "w"):
        pass
    print("🚨 주피터 커널을 자동으로 재시작합니다.")
    print("   재시작이 끝나면 (좌측 상단 커널 상태가 idle이 되면) 이 셀부터 다시 실행해 주세요.")
    IPython.Application.instance().kernel.do_shutdown(True)

### 인증 및 클라이언트 초기화 (Authentication)

Gemini Embedding 2 호출과 Vector Search 2.0 호출 **둘 다** 이 프로젝트의 ADC(Application Default
Credentials)를 사용합니다. Cloud Shell / Workbench에는 이미 자격 증명이 붙어 있으므로
**API Key를 따로 준비할 필요가 없습니다.**

> [!NOTE]
> Gemini Embedding 2는 Vertex AI의 **`global` 엔드포인트**에서 제공됩니다.
> 리전 엔드포인트(`us-central1` 등)로 호출하면 404가 반환되므로,
> 임베딩 클라이언트만 `location="global"` 로 만듭니다.
> 컬렉션 자체는 `us-central1` 에 그대로 둡니다.


In [ ]:
import os
import subprocess
import google.auth
from google import genai


# 1. Project ID 자동 감지
def get_project_id():
    """활성화된 Google Cloud Project ID를 자동으로 탐지합니다."""
    try:
        _, project_id = google.auth.default()
        if project_id:
            return project_id
    except Exception:
        pass
    return subprocess.check_output(["gcloud", "config", "get-value", "project"]).decode("utf-8").strip()


PROJECT_ID = get_project_id()
REGION = "us-central1"                 # Vector Search 2.0 컬렉션 리전
EMBEDDING_LOCATION = "global"          # Gemini Embedding 2 제공 엔드포인트
SOURCE_BUCKET = "ai-multimodal-data"   # 실습용 공용 읽기 전용 버킷
MODEL_ID = "gemini-embedding-2"

# 2. GenAI 클라이언트 — Vertex AI(ADC) 경로. API Key 불필요.
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=EMBEDDING_LOCATION,
)

print(f"Google Cloud Project ID: {PROJECT_ID}")
print(f"컬렉션 리전: {REGION} / 임베딩 엔드포인트: {EMBEDDING_LOCATION} / 모델: {MODEL_ID}")


## 0단계: Vector Search 2.0 컬렉션 만들기

서버리스 컬렉션 생성은 **10초 안팎**이면 끝납니다. 미리 걸어두고 기다리는 요령이 필요 없으니
여기서 만들고 바로 확인한 뒤 다음 단계로 넘어갑니다.

스키마 포인트 두 가지입니다.
1. `description`을 **데이터 필드**로 저장합니다. 값을 채우지도 않는 sparse 벡터 필드 대신, VS2의 `text_search`로 이 필드를 직접 전문검색합니다.
2. `tag`도 데이터 필드입니다. 뒤에서 `{"tag": {"$eq": "Me"}}` 네이티브 필터의 대상이 됩니다.


In [ ]:
import time
from google.cloud import vectorsearch_v1beta as vectorsearch
from google.api_core import exceptions

COLLECTION_ID = "multimodal-media-collection"
COLLECTION_PARENT = f"projects/{PROJECT_ID}/locations/{REGION}"
COLLECTION_NAME = f"{COLLECTION_PARENT}/collections/{COLLECTION_ID}"

vector_search_client = vectorsearch.VectorSearchServiceClient()

# 데이터 필드: description -> text_search 대상, tag -> filter 대상, source -> 크라우딩 필터 기준
data_schema = {
    "type": "object",
    "properties": {
        "description": {"type": "string"},
        "tag": {"type": "string"},
        "media_type": {"type": "number"},
        "source": {"type": "string"},
    },
}

# 벡터 필드: 3072차원 dense 하나.
# vertex_embedding_config를 붙여두면 검색 시 search_text만 넘겨도 서버가 질의 임베딩을 대신 생성합니다.
vector_schema = {
    "content_embedding": vectorsearch.VectorField(
        dense_vector=vectorsearch.DenseVectorField(
            dimensions=3072,
            vertex_embedding_config=vectorsearch.VertexEmbeddingConfig(
                model_id="gemini-embedding-2"
            ),
        )
    )
}

collection_config = vectorsearch.Collection(
    data_schema=data_schema,
    vector_schema=vector_schema,
)

started_at = time.time()
try:
    operation = vector_search_client.create_collection(
        request=vectorsearch.CreateCollectionRequest(
            parent=COLLECTION_PARENT,
            collection_id=COLLECTION_ID,
            collection=collection_config,
        )
    )
    collection = operation.result()
    print(f"✅ 컬렉션 생성 완료 ({time.time() - started_at:.0f}초): {collection.name}")
except exceptions.AlreadyExists:
    collection = vector_search_client.get_collection(name=COLLECTION_NAME)
    print(f"ℹ️ 동일한 ID의 컬렉션이 이미 존재하여 그대로 재사용합니다: {COLLECTION_ID}")

## 1단계: 비디오 전처리 및 세그먼트 분할 (Chunking)

40~60분짜리 영상 하나에 임베딩 하나를 만들면 정보 손실(Embedding Dilution)이 발생합니다.
특정 시점의 세부 시맨틱을 살리려면 **10초 단위 세그먼트**로 잘게 쪼개야 합니다.

**FFmpeg segment muxer + 스트림 복사(`-c copy`)** 는 재인코딩 없이 원본 스트림을 그대로 잘라 붙이므로 수 초 만에 끝납니다.
소스는 Google Cloud x Team USA - Behind the tech 영상(`gs://ai-multimodal-data/team_usa_tech.mp4`)입니다.

In [ ]:
import os
import cv2
import subprocess
import imageio_ffmpeg
from google.cloud import storage

VIDEO_BLOB = "team_usa_tech.mp4"
CHUNK_DURATION = 10   # seconds
CHUNK_DIR = "local_chunks"

# 1. 원본 영상 다운로드 (이미 있으면 건너뜀)
local_video_path = VIDEO_BLOB
if not os.path.exists(local_video_path):
    print(f"비디오 다운로드 중: gs://{SOURCE_BUCKET}/{VIDEO_BLOB} ...")
    storage.Client().bucket(SOURCE_BUCKET).blob(VIDEO_BLOB).download_to_filename(local_video_path)
print(f"원본 영상 준비 완료: {local_video_path}")

# 2. FFmpeg 스트림 복사 분할
os.makedirs(CHUNK_DIR, exist_ok=True)
for f in os.listdir(CHUNK_DIR):
    if f.endswith(".mp4"):
        os.remove(os.path.join(CHUNK_DIR, f))

ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
print("고성능 FFmpeg 스트림 복사 방식으로 비디오 분할(Chunking)을 시작합니다...")
subprocess.run(
    [
        ffmpeg_bin, "-y", "-i", local_video_path,
        "-c", "copy",
        "-f", "segment",
        "-segment_time", str(CHUNK_DURATION),
        "-reset_timestamps", "1",
        os.path.join(CHUNK_DIR, "chunk_%d.mp4"),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True,
)

# 3. 청크별 정밀 메타데이터(시작/종료 시각) 산출
chunk_metadata = []
chunk_files = sorted(
    [f for f in os.listdir(CHUNK_DIR) if f.startswith("chunk_") and f.endswith(".mp4")],
    key=lambda x: int(x.split("_")[1].split(".")[0]),
)
for i, chunk_name in enumerate(chunk_files):
    chunk_file = os.path.join(CHUNK_DIR, chunk_name)
    cap = cv2.VideoCapture(chunk_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = total_frames / fps if fps > 0 and total_frames > 0 else CHUNK_DURATION
    chunk_metadata.append({
        "chunk_id": i,
        "file_path": chunk_file,
        "start_time": i * CHUNK_DURATION,
        "end_time": i * CHUNK_DURATION + duration,
    })

print(f"정밀 메타데이터를 포함한 {len(chunk_metadata)}개의 비디오 청크를 생성했습니다.")
print("메타데이터 샘플:", chunk_metadata[0])

## 2단계: 멀티모달 임베딩 생성 (Generating Embeddings)

`gemini-embedding-2`는 텍스트·이미지·비디오를 **같은 3072차원 공간**으로 보냅니다. 이것이 크로스모달 검색이 가능한 이유입니다.
아래에서 범용 임베딩 함수를 정의하고, 청크 10개를 `ThreadPoolExecutor`로 **병렬 처리**합니다.
(순차 호출이면 3~5분, 병렬이면 20초 내외입니다.)

In [ ]:
from google.genai import types


def generate_multimodal_embedding(content_path, content_type):
    """Gemini Embedding 2로 텍스트/이미지/비디오를 3072차원 벡터로 변환합니다."""
    if content_type == "text":
        contents = content_path
    else:
        mime_type = "video/mp4" if content_type == "video" else "image/jpeg"
        with open(content_path, "rb") as f:
            contents = types.Part.from_bytes(data=f.read(), mime_type=mime_type)

    response = client.models.embed_content(
        model=MODEL_ID,
        contents=contents,
        config=types.EmbedContentConfig(
            output_dimensionality=3072,
            http_options=types.HttpOptions(
                retry_options=types.HttpRetryOptions(
                    attempts=10,
                    initial_delay=1.0,
                    max_delay=3.0,
                )
            ),
        ),
    )
    return response.embeddings[0].values


print("멀티모달 임베딩 생성 함수 정의 완료")

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 8   # 드라이런 실측 최적값. 16으로 올리면 스로틀링으로 오히려 느려집니다
LIMIT = 10        # 로컬 실습용으로 앞쪽 10개 청크만 사용

valid_chunks = chunk_metadata[:LIMIT]
print(f"상위 {len(valid_chunks)}개 비디오 청크의 밀집 임베딩 벡터를 병렬 생성합니다 (max_workers={MAX_WORKERS})...")

start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    chunk_embeddings = list(
        executor.map(lambda item: generate_multimodal_embedding(item["file_path"], "video"), valid_chunks)
    )
for item, embedding in zip(valid_chunks, chunk_embeddings):
    item["dense_embedding"] = embedding

print(f"{len(valid_chunks)}개 청크 임베딩 생성 완료 ({time.time() - start:.1f}초 소요)")
print(f"벡터 차원: {len(valid_chunks[0]['dense_embedding'])}")

### 📊 glass box #1: 코사인 유사도와 t-SNE 시간축 궤적

관리형 벡터 DB에 넣기 전에, 로컬 메모리에서 직접 수학을 돌려 임베딩 공간의 구조에 대한 직관을 만듭니다.
1. **코사인 유사도**: NumPy `dot`으로 직접 계산해 기준 청크 대비 '가장 유사한 장면'과 '가장 이질적인 장면'을 찾습니다.
2. **t-SNE 궤적**: 3072차원을 2차원으로 축소한 뒤, 시간 순서대로 화살표를 연결해 영상의 씬 전환 리듬을 눈으로 봅니다.

In [ ]:
# ==========================================
# 1. Embedding Distance & Similarity
# ==========================================
import io
import subprocess

import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def cosine_similarity(v1, v2):
    """두 벡터의 코사인 유사도를 numpy.dot으로 직접 계산합니다."""
    a = np.array(v1)
    b = np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def extract_first_frame(video_path):
    """FFmpeg으로 첫 프레임을 뽑아 옵니다. 디코딩이 불가능하면 None을 반환합니다.

    OpenCV 내장 FFmpeg은 AV1 등 최신 코덱 디코더를 포함하지 않는 경우가 있어
    (청킹은 -c copy 라 디코딩 없이 동작합니다) 여기서는 별도 FFmpeg 바이너리를 사용합니다.
    """
    result = subprocess.run(
        [
            imageio_ffmpeg.get_ffmpeg_exe(), "-v", "error",
            "-i", video_path, "-frames:v", "1", "-f", "image2", "-c:v", "png", "pipe:1",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    if result.returncode != 0 or not result.stdout:
        return None
    return np.array(Image.open(io.BytesIO(result.stdout)))


def show_chunk_frames(chunks, titles):
    """각 청크의 첫 프레임을 나란히 그려 장면을 눈으로 비교합니다 (비디오 플레이어 대체).

    제목은 matplotlib 기본 폰트에 한글 글리프가 없어 영문으로 표기합니다.
    """
    fig, axes = plt.subplots(1, len(chunks), figsize=(4 * len(chunks), 3))
    skipped = 0
    for ax, chunk, title in zip(np.atleast_1d(axes), chunks, titles):
        frame = extract_first_frame(chunk["file_path"])
        if frame is not None:
            ax.imshow(frame)
        else:
            skipped += 1
            ax.text(0.5, 0.5, "preview unavailable", ha="center", va="center", fontsize=9, color="gray")
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    if skipped:
        print(f"ℹ️ 프레임 미리보기 {skipped}건을 건너뛰었습니다 (코덱 디코딩 미지원). 유사도 수치는 정상입니다.")


print("=== 로컬 임베딩 코사인 유사도 연산 분석 (DB 연결 없이 진행) ===")
ref_chunk = valid_chunks[0]
similarities = sorted(
    [(item, cosine_similarity(ref_chunk["dense_embedding"], item["dense_embedding"])) for item in valid_chunks[1:]],
    key=lambda x: x[1],
    reverse=True,
)

most_similar, max_sim = similarities[0]
least_similar, min_sim = similarities[-1]

print(f"기준 세그먼트 (Reference): Chunk {ref_chunk['chunk_id']} ({ref_chunk['start_time']}초 - {ref_chunk['end_time']:.1f}초)\n")
print(f"✅ [가장 유사한 세그먼트] Chunk {most_similar['chunk_id']} ({most_similar['start_time']}초 - {most_similar['end_time']:.1f}초) | 코사인 유사도 {max_sim:.4f}")
print(f"❌ [가장 이질적인 세그먼트] Chunk {least_similar['chunk_id']} ({least_similar['start_time']}초 - {least_similar['end_time']:.1f}초) | 코사인 유사도 {min_sim:.4f}")

show_chunk_frames(
    [ref_chunk, most_similar, least_similar],
    [
        f"Chunk {ref_chunk['chunk_id']} (reference)",
        f"Chunk {most_similar['chunk_id']} (closest {max_sim:.3f})",
        f"Chunk {least_similar['chunk_id']} (farthest {min_sim:.3f})",
    ],
)

In [ ]:
# ===================================================
# 2. t-SNE Dimensionality Reduction & Trajectory Plot
# ===================================================
import seaborn as sns
from sklearn.manifold import TSNE

print("비디오 청크 임베딩에 대한 t-SNE 차원 축소(3072 -> 2)를 수행 중입니다...")

embeddings_matrix = np.array([item["dense_embedding"] for item in valid_chunks])
chronological_labels = [f"{item['start_time']}s - {item['end_time']:.0f}s" for item in valid_chunks]
chunk_indices = list(range(len(valid_chunks)))

tsne = TSNE(n_components=2, perplexity=min(5, len(valid_chunks) - 1), random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_matrix)

plt.figure(figsize=(10, 8))
sns.set_theme(style="whitegrid")

scatter = plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    c=chunk_indices,
    cmap="viridis",
    s=150,
    zorder=3,
    edgecolors="black",
    linewidth=1.5,
)

# 시간 순서대로 좌표를 연결하여 '비디오 궤적'을 그립니다.
for i in range(len(embeddings_2d) - 1):
    plt.annotate(
        "",
        xy=(embeddings_2d[i + 1, 0], embeddings_2d[i + 1, 1]),
        xytext=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
        arrowprops=dict(arrowstyle="->", color="red", lw=1.5, ls="--", alpha=0.6, connectionstyle="arc3,rad=0.1"),
    )

for idx, (x, y) in enumerate(embeddings_2d):
    plt.text(
        x + 0.2,
        y + 0.2,
        f"Chunk {idx}\n({chronological_labels[idx]})",
        fontsize=9,
        weight="bold",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", edgecolor="gray", alpha=0.8),
        zorder=4,
    )

plt.colorbar(scatter, label="Chronological Chunk Index")
plt.title("Chronological Video Embedding Trajectory (t-SNE Projections)", fontsize=14, weight="bold")
plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.tight_layout()
plt.show()

print("\n💡 해석 가이드:")
print("- 뭉쳐 있는 구간: 장면 흐름이 정적이거나 연속적입니다.")
print("- 크게 도약하는 선: 씬 전환, 페이드, 급격한 카메라 무브먼트가 발생한 지점입니다.")
print("- 이전 좌표로 되돌아오는 궤적: 동일 씬의 반복(예: 스튜디오 화면 복귀)을 암시합니다.")

### 🔍 Dense 단독 시맨틱 검색 — 크로스모달을 체감하기

설명문(텍스트)을 아직 만들지 않은 상태에서, **텍스트 질의 벡터와 비디오 청크 벡터의 거리만으로** 검색해 봅니다.
자막도, 키워드도, 파일명도 쓰지 않습니다. 순수하게 "의미"만으로 영상 구간을 찾아내는지 확인하는 단계입니다.

In [ ]:
DENSE_QUERY = "athletes training in the snow"   # 원하는 질의로 바꿔 보세요 (한국어 질의도 동작합니다)

query_dense = generate_multimodal_embedding(DENSE_QUERY, "text")
dense_scores = [cosine_similarity(query_dense, item["dense_embedding"]) for item in valid_chunks]
ranked_indices = np.argsort(dense_scores)[::-1][:3]

print(f"🔍 [Dense 단독 시맨틱 검색] 질의: '{DENSE_QUERY}'")
print("   설명문 없이 '텍스트 벡터 ↔ 영상 벡터' 거리만으로 매칭합니다.\n")
for rank, idx in enumerate(ranked_indices, start=1):
    item = valid_chunks[idx]
    print(f"[{rank}위] 코사인 유사도 {dense_scores[idx]:.4f} | Chunk {item['chunk_id']} ({item['start_time']}초 - {item['end_time']:.1f}초)")

show_chunk_frames(
    [valid_chunks[idx] for idx in ranked_indices],
    [f"Rank {rank} ({dense_scores[idx]:.3f})" for rank, idx in enumerate(ranked_indices, start=1)],
)

## 3단계: 로컬 하이브리드 검색 구현 (glass box #2)

밀집(Dense) 시맨틱 매칭만으로는 고유명사·숫자·정확한 키워드에 약합니다. 그래서 실무 검색기는 **희소(Sparse) 키워드 점수를 함께 씁니다.**
여기서는 그 파이프라인을 직접 손으로 조립합니다.

1. **Gemini Flash 캡션** — 청크마다 짧은 영어 설명문을 병렬 생성합니다 (희소 검색의 입력 텍스트).
2. **SimpleBM25** — 40줄짜리 순수 파이썬 BM25를 직접 구현합니다.
3. **`alpha` 결합** — `alpha * dense + (1 - alpha) * sparse`. 이 `alpha`가 나중에 VS2의 RRF `weights`로 그대로 대응됩니다.

In [ ]:
def generate_chunk_description(video_path):
    """Gemini Flash로 10초 클립의 짧은 설명문을 생성합니다 (희소/전문 검색용 텍스트)."""
    with open(video_path, "rb") as f:
        part = types.Part.from_bytes(data=f.read(), mime_type="video/mp4")
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=[
            part,
            "Provide a short, detailed description of what is happening in this 10-second video clip. Focus on keywords, actions, and objects.",
        ],
        config=types.GenerateContentConfig(
            http_options=types.HttpOptions(
                retry_options=types.HttpRetryOptions(
                    attempts=10,
                    initial_delay=1.0,
                    max_delay=3.0,
                )
            )
        ),
    )
    return response.text


print(f"Gemini Flash로 {len(valid_chunks)}개 청크의 설명문을 병렬 생성합니다 (max_workers={MAX_WORKERS})...")
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    chunk_descriptions = list(
        executor.map(lambda item: generate_chunk_description(item["file_path"]), valid_chunks)
    )
for item, description in zip(valid_chunks, chunk_descriptions):
    item["description"] = description.strip()

print(f"설명문 생성 완료 ({time.time() - start:.1f}초 소요)\n")
print("샘플 (Chunk 0):", valid_chunks[0]["description"][:200], "...")

In [ ]:
import math
from collections import Counter


class SimpleBM25:
    """교육용 BM25 구현. 검색 엔진의 희소(Sparse) 점수가 어떻게 계산되는지 보여줍니다."""

    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.avgdl = sum(len(doc) for doc in corpus) / self.corpus_size if self.corpus_size > 0 else 0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []

        for doc in corpus:
            self.doc_len.append(len(doc))
            self.doc_freqs.append(Counter(doc))

        for doc in corpus:
            for word in set(doc):
                self.idf[word] = self.idf.get(word, 0) + 1

        for word, freq in self.idf.items():
            self.idf[word] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1)

    def get_scores(self, query):
        scores = []
        for i in range(self.corpus_size):
            score = 0
            doc_freq = self.doc_freqs[i]
            doc_len = self.doc_len[i]
            for word in query:
                if word in doc_freq:
                    idf = self.idf.get(word, 0)
                    freq = doc_freq[word]
                    numerator = freq * (self.k1 + 1)
                    denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                    score += idf * numerator / denominator
            scores.append(score)
        return scores


def tokenize(text):
    return text.lower().split()


bm25 = SimpleBM25([tokenize(item["description"]) for item in valid_chunks])
print(f"{len(valid_chunks)}개 문서 기반 희소 검색용 BM25 인덱스가 빌드되었습니다.")

In [ ]:
def hybrid_search(query_text, alpha=0.7, top_k=3):
    """Dense 코사인 점수와 BM25 희소 점수를 alpha 가중치로 선형 결합합니다.

    alpha=1.0 이면 순수 시맨틱, alpha=0.0 이면 순수 키워드 검색이 됩니다.
    """

    def normalize(scores):
        # 두 점수는 스케일이 완전히 다르므로 [0, 1]로 정규화한 뒤에 더해야 합니다.
        if scores.max() == scores.min():
            return np.zeros_like(scores)
        return (scores - scores.min()) / (scores.max() - scores.min())

    query_embedding = generate_multimodal_embedding(query_text, "text")
    dense_scores = normalize(np.array([cosine_similarity(query_embedding, item["dense_embedding"]) for item in valid_chunks]))
    sparse_scores = normalize(np.array(bm25.get_scores(tokenize(query_text))))
    combined_scores = alpha * dense_scores + (1 - alpha) * sparse_scores

    return [
        {
            "chunk": valid_chunks[idx],
            "score": combined_scores[idx],
            "dense_score": dense_scores[idx],
            "sparse_score": sparse_scores[idx],
        }
        for idx in np.argsort(combined_scores)[::-1][:top_k]
    ]


HYBRID_QUERY = "olympic athlete interview"

for alpha in (0.8, 0.3):
    label = "밀집 시맨틱 중심" if alpha > 0.5 else "희소 키워드 중심"
    print(f"\n=== alpha = {alpha} ({label}) | 질의: '{HYBRID_QUERY}' ===")
    for rank, res in enumerate(hybrid_search(HYBRID_QUERY, alpha=alpha), start=1):
        chunk = res["chunk"]
        print(f"[{rank}위] 최종 {res['score']:.3f} (dense {res['dense_score']:.3f} / sparse {res['sparse_score']:.3f}) | Chunk {chunk['chunk_id']}")
        print(f"       {chunk['description'][:110]}...")

## 4단계: 스케일 업 — 사전 연산 레지스트리 로드 & 개인화 태깅

청크 10개로는 검색 엔진이라 하기 어렵습니다. 실제 코퍼스 규모로 올리기 위해, 미리 임베딩까지 끝내둔 레지스트리를 내려받습니다.
(`full_dataset_registry.pkl` · 135MB · 이미지 4,606 + 비디오 청크 199 · 3072차원)

전량 업서트는 수 분이 걸리므로 **약 1,000건으로 서브샘플링**합니다(비디오 청크 199개 전량 + 이미지 800개).

그리고 개인화 실습을 위해 사진 3장에 `"Me"` 태그를 붙입니다.
기존처럼 설명문 문자열에 `"Me"`를 이어 붙이는 꼼수가 아니라, **독립된 데이터 필드**로 저장합니다.
그래야 뒤에서 VS2 네이티브 필터 `{"tag": {"$eq": "Me"}}`를 쓸 수 있습니다.

In [ ]:
import pickle
import urllib.request
from IPython.display import display, Image as IPImage

REGISTRY_URL = "https://storage.googleapis.com/ai-multimodal-data/full_dataset_registry.pkl"
REGISTRY_FILE = "full_dataset_registry.pkl"
IMAGE_LIMIT = 800   # 업서트할 이미지 수 (비디오 청크는 전량 사용)

TAG_ME = "Me"
TAG_PUBLIC = "Public"
TARGET_IDS = [
    "1003163366_44323f5815.jpg",
    "1007129816_e794419615.jpg",
    "1015118661_980735411b.jpg",
]

if not os.path.exists(REGISTRY_FILE):
    print(f"사전 연산된 레지스트리를 다운로드합니다 (135MB): {REGISTRY_URL}")
    urllib.request.urlretrieve(REGISTRY_URL, REGISTRY_FILE)

with open(REGISTRY_FILE, "rb") as f:
    full_registry = pickle.load(f)
print(f"전체 레지스트리 항목 수: {len(full_registry)}")

# 1. 서브샘플링 (태깅 대상 3개는 반드시 포함되도록 먼저 뽑습니다)
video_items = [item for item in full_registry if item["type"] == "video_chunk"]
tagged_items = [item for item in full_registry if item["id"] in TARGET_IDS]
other_images = [item for item in full_registry if item["type"] == "image" and item["id"] not in TARGET_IDS]
combined_registry = video_items + tagged_items + other_images[: IMAGE_LIMIT - len(tagged_items)]

# 2. 데이터 필드 부여: tag(필터용) / source(크라우딩용) / dp_id(VS2 오브젝트 ID)
for item in combined_registry:
    item["tag"] = TAG_ME if item["id"] in TARGET_IDS else TAG_PUBLIC
    if item["type"] == "video_chunk":
        item["source"] = item.get("source_video", item["id"].rsplit("_", 1)[0])
    else:
        item["source"] = "flickr8k"
    item["dp_id"] = item["id"].replace(".", "_").replace("-", "_").replace(" ", "_")

registry_by_dp_id = {item["dp_id"]: item for item in combined_registry}
image_count = sum(1 for item in combined_registry if item["type"] == "image")
print(f"업서트 대상: 총 {len(combined_registry)}건 (video_chunk {len(video_items)} / image {image_count})")

# 3. 로컬 인메모리 인덱스도 같은 서브샘플로 재구성 (15번 셀의 3-way 비교에 사용)
registry_matrix = np.array([item["dense_embedding"] for item in combined_registry], dtype=np.float32)
registry_matrix /= np.linalg.norm(registry_matrix, axis=1, keepdims=True)
combined_bm25 = SimpleBM25([tokenize(item["description"]) for item in combined_registry])
print(f"인메모리 벡터 행렬 {registry_matrix.shape} + 통합 BM25 인덱스 빌드 완료")


# 4. 결과 렌더러 (거대한 대시보드 대신 이 함수 하나만 사용합니다)
def load_image_bytes(gcs_path):
    bucket_name, blob_name = gcs_path[5:].split("/", 1)
    return storage.Client().bucket(bucket_name).blob(blob_name).download_as_bytes()


def field(data_object, name, default=""):
    return data_object.data[name] if name in data_object.data else default


def vs2_rows(results):
    """Vector Search 2.0 검색 응답을 렌더링용 dict 리스트로 변환합니다."""
    return [
        {
            "id": result.data_object.data_object_id,
            "score": result.distance,
            "description": field(result.data_object, "description"),
            "tag": field(result.data_object, "tag"),
            "media_type": field(result.data_object, "media_type", 1),
            "source": field(result.data_object, "source"),
        }
        for result in results
    ]


def local_rows(items, scores):
    """로컬 검색 결과를 vs2_rows와 동일한 형태로 맞춰 나란히 비교할 수 있게 합니다."""
    return [
        {
            "id": item["dp_id"],
            "score": float(score),
            "description": item["description"],
            "tag": item["tag"],
            "media_type": 1 if item["type"] == "image" else 0,
            "source": item["source"],
        }
        for item, score in zip(items, scores)
    ]


def show_results(rows, title, preview=3):
    """검색 결과를 순위대로 출력하고, 상위 이미지 몇 장만 썸네일로 보여줍니다."""
    print(f"\n[{title}]")
    for rank, row in enumerate(rows, start=1):
        kind = "image" if row["media_type"] == 1 else "video_chunk"
        print(f"  {rank}위 | score {row['score']:.4f} | {row['id']} ({kind}, tag={row['tag']})")
        print(f"        {row['description'][:100]}...")

    for row in [row for row in rows if row["media_type"] == 1][:preview]:
        item = registry_by_dp_id.get(row["id"])
        if item:
            display(IPImage(data=load_image_bytes(item["path"]), width=180))


print(f"\n🏷️ '{TAG_ME}' 태그가 부착된 개인 사진 {len(tagged_items)}장:")
for item in tagged_items:
    print(f"  - {item['id']}: {item['description'][:80]}...")
    display(IPImage(data=load_image_bytes(item["path"]), width=180))

### 병렬 배치 업서트

`BatchCreateDataObjects`의 1회 요청 상한은 **250건**이므로 여러 배치로 나눠 병렬 전송합니다.

> Part 1에서는 **ANN 인덱스를 만들지 않습니다.** 인덱스가 없어도 VS2는 kNN 완전탐색으로 검색·필터·전문검색을 모두 수행합니다.
> (수만 건 규모에서는 인덱스 생성에 수십 분이 걸리며, Part 2에서 ANN이 걸린 컬렉션과 대조하게 됩니다.)


In [ ]:
UPSERT_WORKERS = 8   # 배치 전송 동시성 (드라이런 실측 최적값)
BATCH_SIZE = 250     # BatchCreateDataObjects 1회 요청 상한

data_client = vectorsearch.DataObjectServiceClient()
search_client = vectorsearch.DataObjectSearchServiceClient()

# 1. 업서트 요청 조립
upsert_requests = [
    vectorsearch.CreateDataObjectRequest(
        parent=collection.name,
        data_object_id=item["dp_id"],
        data_object=vectorsearch.DataObject(
            data={
                "description": item["description"],
                "tag": item["tag"],
                "media_type": 1 if item["type"] == "image" else 0,
                "source": item["source"],
            },
            vectors={
                "content_embedding": vectorsearch.Vector(
                    dense=vectorsearch.DenseVector(values=item["dense_embedding"])
                )
            },
        ),
    )
    for item in combined_registry
]
batches = [upsert_requests[i : i + BATCH_SIZE] for i in range(0, len(upsert_requests), BATCH_SIZE)]


def send_batch(batch):
    try:
        data_client.batch_create_data_objects(
            request=vectorsearch.BatchCreateDataObjectsRequest(parent=collection.name, requests=batch)
        )
    except exceptions.AlreadyExists:
        pass   # 노트북 재실행 시 이미 올라간 오브젝트는 건너뜁니다
    return len(batch)


print(f"{len(batches)}개 배치를 병렬 업서트합니다 (max_workers={UPSERT_WORKERS})...")
start = time.time()
with ThreadPoolExecutor(max_workers=UPSERT_WORKERS) as executor:
    upserted = sum(executor.map(send_batch, batches))
print(f"{upserted}개 데이터 오브젝트 업서트 완료 ({time.time() - start:.1f}초 소요)")


### ⭐ [핵심] 같은 질의를 3-way로 — 로컬 완전탐색 / 로컬 하이브리드 / Vector Search 2.0

지금까지 손으로 짠 것과 매니지드 서비스가 **같은 수학**을 한다는 것을 한 셀에서 확인합니다.

| | 검색 주체 | 질의 임베딩 | 유사도 연산 |
| :--- | :--- | :--- | :--- |
| 로컬 완전탐색 | 내 노트북 NumPy | 내가 API 호출 | `matrix @ query` |
| 로컬 alpha 하이브리드 | 내 노트북 NumPy + BM25 | 위 결과 재사용 | 정규화 후 가중합 |
| **Vector Search 2.0** | **매니지드 서버리스 엔진** | **서버가 대신 생성** | **kNN 완전탐색** |

In [ ]:
COMPARE_QUERY = "a dog running on the beach"

# 1) 로컬 완전탐색 (dense only) — 질의 임베딩 API 호출 + 1,000 x 3072 행렬곱
start = time.time()
query_vector = np.array(generate_multimodal_embedding(COMPARE_QUERY, "text"), dtype=np.float32)
query_vector /= np.linalg.norm(query_vector)
dense_registry_scores = registry_matrix @ query_vector
top_dense = np.argsort(dense_registry_scores)[::-1][:5]
local_dense_ms = (time.time() - start) * 1000

# 2) 로컬 alpha 하이브리드 — 위에서 만든 임베딩을 재사용하므로 API 호출 없음
start = time.time()
sparse_registry_scores = np.array(combined_bm25.get_scores(tokenize(COMPARE_QUERY)))
if sparse_registry_scores.max() > 0:
    sparse_registry_scores = sparse_registry_scores / sparse_registry_scores.max()
dense_normalized = (dense_registry_scores - dense_registry_scores.min()) / (
    dense_registry_scores.max() - dense_registry_scores.min()
)
ALPHA = 0.7
local_hybrid_scores = ALPHA * dense_normalized + (1 - ALPHA) * sparse_registry_scores
top_hybrid = np.argsort(local_hybrid_scores)[::-1][:5]
local_hybrid_ms = (time.time() - start) * 1000

# 3) Vector Search 2.0 kNN — 질의 텍스트만 던지면 임베딩부터 검색까지 서버가 처리
start = time.time()
vs2_results = search_client.search_data_objects(
    vectorsearch.SearchDataObjectsRequest(
        parent=collection.name,
        semantic_search=vectorsearch.SemanticSearch(
            search_text=COMPARE_QUERY,
            search_field="content_embedding",
            task_type="QUESTION_ANSWERING",
            top_k=5,
            output_fields=vectorsearch.OutputFields(
                data_fields=["description", "tag", "media_type", "source"]
            ),
        ),
    )
)
rows_vs2 = vs2_rows(vs2_results)
vs2_ms = (time.time() - start) * 1000

print(f"질의: '{COMPARE_QUERY}' | 코퍼스 {len(combined_registry)}건\n")
print(f"  로컬 완전탐색 (NumPy)        : {local_dense_ms:8.1f} ms  (질의 임베딩 API 호출 포함)")
print(f"  로컬 alpha={ALPHA} 하이브리드   : {local_hybrid_ms:8.1f} ms  (임베딩 재사용, 순수 CPU 연산)")
print(f"  Vector Search 2.0 kNN        : {vs2_ms:8.1f} ms  (임베딩 생성까지 서버가 수행)")

show_results(
    local_rows([combined_registry[i] for i in top_dense], [dense_registry_scores[i] for i in top_dense]),
    "① 로컬 완전탐색 (NumPy 코사인)",
    preview=0,
)
show_results(
    local_rows([combined_registry[i] for i in top_hybrid], [local_hybrid_scores[i] for i in top_hybrid]),
    f"② 로컬 alpha={ALPHA} 하이브리드 (NumPy + SimpleBM25)",
    preview=0,
)
show_results(rows_vs2, "③ Vector Search 2.0 semantic_search (kNN)", preview=3)

### 내 `alpha`는 곧 VS2의 RRF `weights`

12번 셀에서 손으로 짠 결합식을 기억하시나요?

```python
combined = alpha * dense_scores + (1 - alpha) * sparse_scores   # 내가 짠 것
```

Vector Search 2.0에서는 `batch_search_data_objects`에 검색 절(clause) 두 개를 넣고, 융합 방식을 선언만 하면 됩니다.

```python
ranker=Ranker(rrf=ReciprocalRankFusion(weights=[dense_weight, sparse_weight]))   # 선언으로 대체
```

차이는 두 가지입니다. **(1)** 점수 정규화를 내가 안 해도 됩니다. RRF는 점수가 아니라 **순위(rank)** 로 융합하므로 스케일 문제가 원천적으로 없습니다.
**(2)** 희소 쪽이 내가 만든 BM25가 아니라, `description` 데이터 필드에 대한 VS2 네이티브 `text_search`입니다. (별도 sparse 벡터 필드가 필요 없습니다.)

In [ ]:
OUTPUT_FIELDS = vectorsearch.OutputFields(data_fields=["description", "tag", "media_type", "source"])


def rrf_search(query_text, dense_weight, sparse_weight, top_k=5):
    """semantic_search(밀집) + text_search(전문검색)를 VS2 내장 RRF로 융합합니다."""
    batch_search_request = vectorsearch.BatchSearchDataObjectsRequest(
        parent=collection.name,
        searches=[
            # A. Dense Semantic Search
            vectorsearch.Search(
                semantic_search=vectorsearch.SemanticSearch(
                    search_text=query_text,
                    search_field="content_embedding",
                    task_type="QUESTION_ANSWERING",
                    top_k=20,
                    output_fields=OUTPUT_FIELDS,
                )
            ),
            # B. Sparse Full-Text Search (description 데이터 필드 직접 검색)
            vectorsearch.Search(
                text_search=vectorsearch.TextSearch(
                    search_text=query_text,
                    data_field_names=["description"],
                    top_k=20,
                    output_fields=OUTPUT_FIELDS,
                )
            ),
        ],
        combine=vectorsearch.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch.Ranker(
                rrf=vectorsearch.ReciprocalRankFusion(weights=[dense_weight, sparse_weight])
            )
        ),
    )
    response = search_client.batch_search_data_objects(batch_search_request)
    # ranker를 지정하면 결과가 하나의 통합 랭킹 리스트로 돌아옵니다.
    if not response.results:
        print("⚠️ 융합 결과가 비어 있습니다. 업서트 셀이 정상적으로 끝났는지 확인하세요.")
        return []
    return vs2_rows(response.results[0].results[:top_k])


RRF_QUERY = "children playing in the water"

for dense_weight, sparse_weight in [(0.8, 0.2), (0.2, 0.8)]:
    rows = rrf_search(RRF_QUERY, dense_weight, sparse_weight)
    show_results(rows, f"RRF weights = [dense {dense_weight}, sparse {sparse_weight}] | 질의: '{RRF_QUERY}'", preview=2)

### 🏷️ 개인화: 네이티브 메타데이터 필터

13번 셀에서 사진 3장에 `tag="Me"`를 데이터 필드로 붙여 두었습니다.
이제 설명문에 `"Me"`라는 단어를 억지로 끼워 넣는 꼼수 없이, **필터 절**로 검색 범위를 좁힙니다.

필터 문법은 MongoDB 스타일 JSON이며(`$eq`, `$lt`, `$in`, `$and` ...), **각 검색 절(clause)에 개별로** 붙습니다.

In [ ]:
def semantic_search(query_text, top_k=5, metadata_filter=None):
    """VS2 시맨틱 검색. metadata_filter는 MongoDB 스타일 JSON입니다."""
    search_kwargs = {
        "search_text": query_text,
        "search_field": "content_embedding",
        "task_type": "QUESTION_ANSWERING",
        "top_k": top_k,
        "output_fields": OUTPUT_FIELDS,
    }
    if metadata_filter:
        search_kwargs["filter"] = metadata_filter

    return vs2_rows(
        search_client.search_data_objects(
            vectorsearch.SearchDataObjectsRequest(
                parent=collection.name,
                semantic_search=vectorsearch.SemanticSearch(**search_kwargs),
            )
        )
    )


FILTER_QUERY = "beach"

show_results(
    semantic_search(FILTER_QUERY, top_k=5),
    f"필터 없음 — 코퍼스 {len(combined_registry)}건 전체 대상 | 질의: '{FILTER_QUERY}'",
    preview=2,
)
show_results(
    semantic_search(FILTER_QUERY, top_k=5, metadata_filter={"tag": {"$eq": TAG_ME}}),
    f"filter={{'tag': {{'$eq': '{TAG_ME}'}}}} — 내 사진 안에서만 검색",
    preview=3,
)

## 5단계: 검색 결과 최적화 (Post-Search Optimization)

1차 검색이 돌려준 후보군을 비즈니스 요구사항에 맞게 보정하는 단계입니다.

* **Ranking API 리랭킹** — 벡터 거리만으로는 잡기 힘든 미세한 질의-문서 적합도를 매니지드 크로스 인코더가 재점수화합니다.
* **비디오 크라우딩 필터** — 같은 영상에서 나온 청크가 상위를 독점하지 않도록 출처별 노출 상한을 겁니다.

In [ ]:
from google.cloud import discoveryengine_v1 as discoveryengine


def rerank_results(query, rows):
    """Agent Search Ranking API로 1차 검색 후보군을 재점수화합니다."""
    print(f"Ranking API에 상위 {len(rows)}개 후보군을 전송하여 재정렬을 요청합니다...")
    rank_client = discoveryengine.RankServiceClient()
    response = rank_client.rank(
        discoveryengine.RankRequest(
            ranking_config=f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config",
            query=query,
            records=[
                discoveryengine.RankingRecord(
                    id=row["id"],
                    title=row["source"],
                    content=row["description"],
                )
                for row in rows
            ],
        )
    )
    rows_by_id = {row["id"]: row for row in rows}
    return [
        dict(rows_by_id[record.id], score=record.score)
        for record in response.records
        if record.id in rows_by_id
    ]


def apply_video_crowding(rows, max_per_video=2):
    """동일 비디오 출처의 청크가 상위를 독점하지 않도록 출처별 노출 수를 제한합니다."""
    filtered_rows = []
    video_counts = {}
    for row in rows:
        if row["media_type"] == 0:
            video_counts[row["source"]] = video_counts.get(row["source"], 0) + 1
            if video_counts[row["source"]] > max_per_video:
                print(f"크라우딩 필터: 출처 {row['source']}의 {row['id']} 노출 제외 (상한 {max_per_video}개 도달)")
                continue
        filtered_rows.append(row)
    return filtered_rows


RERANK_QUERY = "weather prediction machine learning model"

candidates = semantic_search(RERANK_QUERY, top_k=10)
show_results(candidates, f"① 1차 검색 (VS2 semantic_search top 10) | 질의: '{RERANK_QUERY}'", preview=0)

reranked_results = rerank_results(RERANK_QUERY, candidates)
show_results(reranked_results, "② Ranking API 리랭킹 후 (순위 변동 확인)", preview=0)

final_results = apply_video_crowding(reranked_results, max_per_video=2)
show_results(final_results, f"③ 비디오 크라우딩 필터 적용 후 (최종 {len(final_results)}건)", preview=2)

## 정리: 직접 짠 것 ↔ Vector Search 2.0 ↔ Part 2

| 개념 | Part 1에서 직접 짠 것 | Vector Search 2.0 | Part 2에서 |
| :--- | :--- | :--- | :--- |
| 유사도 연산 | NumPy 코사인 완전탐색 | `semantic_search` (kNN) | ANN / ScaNN 인덱스 |
| 키워드 검색 | `SimpleBM25` | `text_search(data_field_names=[...])` | 동일 |
| 가중치 결합 | `alpha * dense + (1-alpha) * sparse` | `ReciprocalRankFusion(weights=[...])` | 가중치 반전 실험 |
| 개인화 | 설명문에 태그 문자열 결합 | `filter={"tag": {"$eq": "Me"}}` | 상품 속성 필터 |
| 리랭킹 | — | Ranking API | 에이전트 런타임에서 동일 API |
| 크로스모달 | 텍스트 질의 ➔ 비디오 청크 | 동일 | 이미지 질의 ➔ 상품 카탈로그 |

**직접 짜본 것이 곧 클라우드 서비스의 내부이고, 그것이 곧 Part 2에서 배포할 에이전트의 검색 엔진입니다.**

## 6단계: 자원 해제 및 청소 (Resource Cleanup)

실습이 끝난 후 불필요한 과금을 예방하기 위해 Vector Search 2.0 서버리스 컬렉션을 삭제합니다.
실수로 `Run All`을 눌러 인덱스가 날아가는 사고를 막기 위해, 이 셀은 **실행되지 않는 Raw 타입**으로 지정되어 있습니다.
정리할 때는 셀 타입을 `Code`로 바꾼 뒤 실행해 주세요.

> ⚠️ Part 2에서는 별도의 상품 컬렉션(`amazon-product-768-compact`)을 사용하므로, 이 셀은 워크숍 **맨 마지막**에 실행하면 됩니다.